<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Efficient_Data_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Working with torch.utils.data.Dataset

In [1]:
import torch
from torch.utils.data import Dataset
import numpy as np

class SimpleCustomDataset(Dataset):
    """A simple example dataset with features and labels."""

    def __init__(self, features, labels):
        """
        Args:
            features (list or np.array): A list or array of features.
            labels (list or np.array): A list or array of labels.
        """
        # Basic check: Features and labels must have the same length
        assert len(features) == len(labels), "Features and labels must have the same length."
        self.features = features
        self.labels = labels

    def __len__(self):
        """Returns the total number of samples."""
        return len(self.features)

    def __getitem__(self, idx):
        """
        Generates one sample of data.

        Args:
            idx (int): The index of the item.

        Returns:
            tuple: (feature, label) for the given index.
        """
        # Retrieve feature and label for the given index
        feature = self.features[idx]
        label = self.labels[idx]

        # Often, you'll convert data to PyTorch tensors here
        # We assume features/labels might not be tensors yet
        sample = (torch.tensor(feature, dtype=torch.float32),
                  torch.tensor(label, dtype=torch.long)) # Assuming classification label

        return sample

# --- Example Usage ---
# Sample data (replace with your actual data)
num_samples = 100
num_features = 10
features_data = np.random.randn(num_samples, num_features)
labels_data = np.random.randint(0, 5, size=num_samples) # Example: 5 classes

# Create an instance of the custom dataset
my_dataset = SimpleCustomDataset(features_data, labels_data)

# Access dataset properties and elements
print(f"Dataset size: {len(my_dataset)}")

# Get the first sample
first_sample = my_dataset[0]
feature_sample, label_sample = first_sample
print(f"\nFirst sample features:\n{feature_sample}")
print(f"First sample shape: {feature_sample.shape}")
print(f"First sample label: {label_sample}")

# Get the tenth sample
tenth_sample = my_dataset[9]
print(f"\nTenth sample label: {tenth_sample[1]}")


Dataset size: 100

First sample features:
tensor([-0.6229, -1.3840,  2.0993, -0.5214,  0.2102,  0.0153,  0.2390, -1.1192,
         0.6814,  0.3199])
First sample shape: torch.Size([10])
First sample label: 3

Tenth sample label: 0


In [5]:
first_sample
#returns a tuple

(tensor([-0.6229, -1.3840,  2.0993, -0.5214,  0.2102,  0.0153,  0.2390, -1.1192,
          0.6814,  0.3199]),
 tensor(3))


# Built-in Datasets (e.g., TorchVision)

In [ ]:
import torchvision
import torchvision.transforms as transforms

# Define a simple transformation to convert images to PyTorch Tensors
transform = transforms.Compose([transforms.ToTensor()])

# Load the training dataset
# root: directory where data will be stored/found
# train=True: specifies the training set
# download=True: downloads the data if not found locally
# transform: applies the defined transformation to each image
train_dataset = torchvision.datasets.CIFAR10(root='./data',
                                             train=True,
                                             download=True,
                                             transform=transform)

# Load the test dataset
test_dataset = torchvision.datasets.CIFAR10(root='./data',
                                            train=False,
                                            download=True,
                                            transform=transform)

print(f"CIFAR-10 training dataset size: {len(train_dataset)}")
print(f"CIFAR-10 test dataset size: {len(test_dataset)}")

# Accessing a single data point (image, label)
img, label = train_dataset[0]
print(f"Image shape: {img.shape}") # Output typically: torch.Size([3, 32, 32])
print(f"Label: {label}")          # Output: An integer representing the class




# Data Transformations (torchvision.transforms)

In [6]:
import torchvision.transforms as transforms

# Example transform pipeline for training
train_transform = transforms.Compose([
    transforms.Resize(256),             # Resize smaller edge to 256
    transforms.RandomCrop(224),         # Randomly crop a 224x224 patch
    transforms.RandomHorizontalFlip(),  # Randomly flip horizontally
    transforms.ToTensor(),              # Convert PIL Image to tensor (0-1 range)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # Normalize with ImageNet stats
                         std=[0.229, 0.224, 0.225])
])

# Example transform pipeline for validation/testing (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),         # Center crop to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Training Transforms:")
print(train_transform)

print("\nTesting Transforms:")
print(test_transform)


Training Transforms:
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    RandomCrop(size=(224, 224), padding=None)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Testing Transforms:
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


# Using torch.utils.data.DataLoader

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

# Assume 'YourCustomDataset' is defined as shown previously
# Or use a built-in dataset like datasets.MNIST
# For demonstration, let's create a simple dummy dataset:
class DummyDataset(Dataset):
    def __init__(self, num_samples=100):
        self.num_samples = num_samples
        self.features = torch.randn(num_samples, 10) # Example: 100 samples, 10 features
        self.labels = torch.randint(0, 2, (num_samples,)) # Example: 100 binary labels

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Instantiate the dataset
dataset = DummyDataset()

# Instantiate the DataLoader
# batch_size: Number of samples per batch
# shuffle: Set to True to shuffle data every epoch (important for training)
train_loader = DataLoader(dataset=dataset, batch_size=32, shuffle=True)

# Iterate over the DataLoader
print(f"Dataset size: {len(dataset)}")
print(f"DataLoader batch size: {train_loader.batch_size}")

for epoch in range(1): # Example for one epoch
    print(f"\n--- Epoch {epoch+1} ---")
    for i, batch in enumerate(train_loader):
        # DataLoader yields batches. Each 'batch' is typically a tuple or list
        # containing tensors for features and labels.
        features, labels = batch
        print(f"Batch {i+1}: Features shape={features.shape}, Labels shape={labels.shape}")
        # Here you would typically perform your training steps:
        # model.train()
        # optimizer.zero_grad()
        # outputs = model(features)
        # loss = criterion(outputs, labels)
        # loss.backward()
        # optimizer.step()


Dataset size: 100
DataLoader batch size: 32

--- Epoch 1 ---
Batch 1: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])
Batch 2: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])
Batch 3: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])
Batch 4: Features shape=torch.Size([4, 10]), Labels shape=torch.Size([4])


In [9]:
# Drop the last incomplete batch if dataset size is not divisible by batch size
train_loader_drop_last = DataLoader(dataset=dataset, batch_size=32, shuffle=True, drop_last=True)

print("\n--- DataLoader with drop_last=True ---")
for i, batch in enumerate(train_loader_drop_last):
    features, labels = batch
    print(f"Batch {i+1}: Features shape={features.shape}, Labels shape={labels.shape}")

# Expected output: Only 3 batches of size 32. The last 9 samples are dropped.



--- DataLoader with drop_last=True ---
Batch 1: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])
Batch 2: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])
Batch 3: Features shape=torch.Size([32, 10]), Labels shape=torch.Size([32])


# Customizing Dataloader behaviour

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Assume 'dataset' is your torch.utils.data.Dataset instance
# Assume 'targets' is a list or tensor containing the class label for each sample
# e.g., targets = [0, 0, 1, 0, ..., 1, 0]

# Calculate weights for each sample
class_counts = torch.bincount(torch.tensor(targets)) # Counts per class: e.g., [900, 100]
num_samples = len(targets) # Total samples: 1000

# Weight for each sample is 1 / (number of samples in its class)
sample_weights = torch.tensor([1.0 / class_counts[t] for t in targets])

# Create the sampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=num_samples, replacement=True)

# Create the DataLoader using the custom sampler
# Note: shuffle must be False when using a sampler
dataloader = DataLoader(dataset, batch_size=32, sampler=sampler)

# Now, batches drawn from this dataloader will have a more balanced
# representation of classes over time.
# for batch_features, batch_labels in dataloader:
#     # Training steps...
#     pass


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Example Dataset returning variable-length tensors
class VariableSequenceDataset(Dataset):
    def __init__(self, data):
        # data is a list of tensors, e.g., [torch.randn(5), torch.randn(8), ...]
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # For simplicity, assume each item also has a label (e.g., its length)
        sequence = self.data[idx]
        label = len(sequence)
        return sequence, label

# Custom collate function
def pad_collate(batch):
    # batch is a list of tuples: [(sequence1, label1), (sequence2, label2), ...]
    # Sort batch elements by sequence length (optional, but often done for RNN efficiency)
    # batch.sort(key=lambda x: len(x[0]), reverse=True) # Not strictly necessary for padding

    # Separate sequences and labels
    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    # Pad sequences to the length of the longest sequence in the batch
    # `batch_first=True` makes the output shape (batch_size, max_seq_len, features)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0.0)

    # Stack labels (assuming they are simple scalars)
    labels = torch.tensor(labels)

    return padded_sequences, labels

# Create dataset and dataloader
sequences = [torch.randn(torch.randint(5, 15, (1,)).item()) for _ in range(100)]
dataset = VariableSequenceDataset(sequences)

dataloader = DataLoader(dataset, batch_size=4, collate_fn=pad_collate)

# Iterate through the dataloader
# for padded_batch, label_batch in dataloader:
#     # padded_batch shape: (4, max_len_in_this_batch, 1) if sequences were 1D
#     # label_batch shape: (4,)
#     # Model processing...
#     pass


# Hands-on Practical: Creating a Data Pipeline


In [11]:
import torch
import torch.utils.data as data
from torchvision import transforms

# Generate synthetic data
num_samples = 100
num_features = 10

# Create random feature vectors (e.g., sensor readings)
features = torch.randn(num_samples, num_features)

# Create random binary labels (0 or 1)
labels = torch.randint(0, 2, (num_samples,))

print(f"Shape of features: {features.shape}") # Output: torch.Size([100, 10])
print(f"Shape of labels: {labels.shape}")   # Output: torch.Size([100])
print(f"First 5 features:\n{features[:5]}")
print(f"First 5 labels:\n{labels[:5]}")


Shape of features: torch.Size([100, 10])
Shape of labels: torch.Size([100])
First 5 features:
tensor([[-1.3814, -0.4230, -2.1815,  0.1374,  1.4479,  0.6751, -1.2534, -0.1733,
          0.3274, -1.2902],
        [ 0.4829,  0.5417, -1.4309, -0.2667,  0.1189,  1.4226,  0.7489,  0.2634,
         -1.0743, -1.3084],
        [-0.0414, -0.8348,  1.5454,  0.3391, -0.0471,  1.5067,  0.4572,  0.8835,
         -1.8343, -0.0198],
        [-0.1070,  0.6815, -0.0570, -1.1771, -1.1482,  0.1939,  1.1198,  0.4608,
          1.7250,  0.8417],
        [ 1.5183, -0.3908, -1.4619, -0.5382, -0.4679, -0.5329, -0.7161,  1.1838,
         -1.1981,  1.3310]])
First 5 labels:
tensor([1, 1, 1, 0, 1])


In [12]:
class SyntheticDataset(data.Dataset):
    """A custom Dataset for our synthetic features and labels."""

    def __init__(self, features, labels, transform=None):
        """
        Args:
            features (Tensor): Tensor containing the feature data.
            labels (Tensor): Tensor containing the labels.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        # Basic check to ensure features and labels have the same number of samples
        assert features.shape[0] == labels.shape[0], \
            "Features and labels must have the same number of samples"

        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        """Returns the total number of samples."""
        return self.features.shape[0]

    def __getitem__(self, idx):
        """
        Retrieves the feature vector and label for a given index.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            tuple: (feature, label) where feature is the feature vector
                   and label is the corresponding label.
        """
        # Get the raw feature and label
        feature_sample = self.features[idx]
        label_sample = self.labels[idx]

        # Create a sample dictionary (or tuple)
        sample = {'feature': feature_sample, 'label': label_sample}

        # Apply transformations if they exist
        if self.transform:
            sample = self.transform(sample)

        # Return the potentially transformed sample
        # Common practice is to return features and labels separately
        return sample['feature'], sample['label']

# Instantiate the dataset without transforms for now
raw_dataset = SyntheticDataset(features, labels)

# Test retrieving a sample
sample_idx = 0
feature_sample, label_sample = raw_dataset[sample_idx]
print(f"\nSample {sample_idx} - Feature: {feature_sample}")
print(f"Sample {sample_idx} - Label: {label_sample}")
print(f"Dataset length: {len(raw_dataset)}") # Output: 100



Sample 0 - Feature: tensor([-1.3814, -0.4230, -2.1815,  0.1374,  1.4479,  0.6751, -1.2534, -0.1733,
         0.3274, -1.2902])
Sample 0 - Label: 1
Dataset length: 100


In [14]:
# Calculate mean and std deviation for normalization (across the dataset)
feature_mean = features.mean(dim=0)
feature_std = features.std(dim=0)

# Avoid division by zero if std dev is zero for any feature
feature_std[feature_std == 0] = 1.0

# Define custom transform classes/functions for our dictionary sample format
class ToTensorAndType(object):
    """Converts features to FloatTensor and labels to LongTensor."""
    def __call__(self, sample):
        feature, label = sample['feature'], sample['label']
        return {'feature': feature.float(), 'label': label.long()}

class NormalizeFeatures(object):
    """Normalizes the feature tensor."""
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, sample):
        feature, label = sample['feature'], sample['label']
        # Apply normalization: (tensor - mean) / std
        normalized_feature = (feature - self.mean) / self.std
        return {'feature': normalized_feature, 'label': label}

# Compose the transformations
data_transforms = transforms.Compose([
    ToTensorAndType(),
    NormalizeFeatures(mean=feature_mean, std=feature_std)
])

# Instantiate the dataset WITH the transformations
transformed_dataset = SyntheticDataset(features, labels, transform=data_transforms)

# Test retrieving a transformed sample
sample_idx = 0
transformed_feature, transformed_label = transformed_dataset[sample_idx]

print(f"\n--- Transformed Sample {sample_idx} ---")
print(f"Original Feature:\n{features[sample_idx]}")
print(f"Transformed Feature:\n{transformed_feature}")
print(f"Original Label: {labels[sample_idx]} (dtype={labels.dtype})")
print(f"Transformed Label: {transformed_label} (dtype={transformed_label.dtype})")
print(f"mean:{feature_mean}")
print(f"std:{feature_std}")
# Verify normalization (mean should be close to 0, std close to 1 for the first sample's features)
print(f"Transformed Feature Mean: {transformed_feature.mean():.4f}") # Should be near 0 after normalization applied across dataset



--- Transformed Sample 0 ---
Original Feature:
tensor([-1.3814, -0.4230, -2.1815,  0.1374,  1.4479,  0.6751, -1.2534, -0.1733,
         0.3274, -1.2902])
Transformed Feature:
tensor([-1.5193, -0.1176, -1.9619,  0.0578,  1.6392,  0.5917, -1.2643, -0.1033,
         0.3041, -1.4071])
Original Label: 1 (dtype=torch.int64)
Transformed Label: 1 (dtype=torch.int64)
mean:tensor([ 0.0230, -0.3030, -0.1304,  0.0733, -0.0649,  0.0811,  0.0664, -0.0744,
         0.0111, -0.0511])
std:tensor([0.9244, 1.0205, 1.0455, 1.1104, 0.9229, 1.0039, 1.0438, 0.9571, 1.0400,
        0.8806])
Transformed Feature Mean: -0.3781


In [15]:
# Create the DataLoader
batch_size = 16 # Process data in batches of 16 samples
shuffle_data = True # Shuffle the data at the beginning of each epoch
num_workers = 0 # Number of subprocesses to use for data loading. 0 means data loading happens in the main process.

# On platforms other than Windows, you can often set num_workers > 0 for parallel loading
# import os
# if os.name != 'nt': # Check if not Windows
#     num_workers = 2

data_loader = data.DataLoader(
    transformed_dataset,
    batch_size=batch_size,
    shuffle=shuffle_data,
    num_workers=num_workers
)

# Iterate through the DataLoader to get batches
print(f"\n--- Iterating through DataLoader (batch_size={batch_size}) ---")

# Get one batch
feature_batch, label_batch = next(iter(data_loader))

print(f"Type of feature_batch: {type(feature_batch)}")
print(f"Shape of feature_batch: {feature_batch.shape}") # Output: torch.Size([16, 10])
print(f"Shape of label_batch: {label_batch.shape}")   # Output: torch.Size([16])
print(f"Data type of feature_batch: {feature_batch.dtype}") # Output: torch.float32
print(f"Data type of label_batch: {label_batch.dtype}")   # Output: torch.int64

# You can loop through all batches like this (e.g., in a training epoch)
# print("\nLooping through a few batches:")
# for i, (batch_features, batch_labels) in enumerate(data_loader):
#     if i >= 3: # Show first 3 batches
#         break
#     print(f"Batch {i+1}: Features shape={batch_features.shape}, Labels shape={batch_labels.shape}")
#     # In a real training loop, you would feed batch_features to your model here



--- Iterating through DataLoader (batch_size=16) ---
Type of feature_batch: <class 'torch.Tensor'>
Shape of feature_batch: torch.Size([16, 10])
Shape of label_batch: torch.Size([16])
Data type of feature_batch: torch.float32
Data type of label_batch: torch.int64
